In [ ]:
import glob, json, os, pathlib, shutil, subprocess, sys
base = pathlib.Path('/kaggle/working/cloud-check')
base.mkdir(parents=True, exist_ok=True)
source = pathlib.Path(glob.glob('/kaggle/input/**/src/kaggle_main.cpp', recursive=True)[0]).parent.parent
shutil.copytree(source / 'include', base / 'include/core', dirs_exist_ok=True)
files = {'scripts/cloud_setup.py': '"""Build the CPU-only inference readout tool for Linux cloud development.\n\nRun from any directory: python3 scripts/cloud_setup.py\nUse --build-only when puzzle data is supplied separately or networking is off.\n"""\nimport argparse\nimport os\nfrom pathlib import Path\nimport shlex\nimport shutil\nimport subprocess\nimport sys\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--build-only", action="store_true", help="do not fetch puzzle data")\n    args = parser.parse_args()\n    root = Path(__file__).resolve().parent.parent\n    compiler = shlex.split(os.environ.get("CXX", "g++"))\n    if not compiler or not shutil.which(compiler[0]):\n        parser.error("a C++20 compiler is needed: install g++ or set CXX=clang++")\n    out = root / "out" / "cloud"\n    out.mkdir(parents=True, exist_ok=True)\n    command = compiler + [\n        "-std=c++20", "-O2", "-pthread", "-DARC_ENABLE_CUDA_PIECES=0",\n        "-I", str(root / "include" / "core"),\n        str(root / "scripts" / "probe.cpp"), "-o", str(out / "arc_probe"),\n    ]\n    print(shlex.join(command), flush=True)\n    subprocess.run(command, cwd=root, check=True)\n    if not args.build_only:\n        data = root / "kaggle/input/competitions/arc-prize-2026-arc-agi-2"\n        needed = ("arc-agi_training_challenges.json", "arc-agi_training_solutions.json")\n        if not all((data / name).is_file() for name in needed):\n            subprocess.run([sys.executable, str(root / "scripts/fetch_data.py")], cwd=root, check=True)\n    print("CPU inference build ready. From the repo root:")\n    print("  out/cloud/arc_probe out/probe/cloud 00576224")\n    print("Readouts only; arc_said and CopyTo still require the CUDA build.")\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/probe.cpp': '// EVERY READOUT, WRITTEN DOWN. Infers every training puzzle -- or only the ids\n// named -- and writes each one\'s inference readout to <out-dir>/logs/inference/\n// <id>.txt, plus one line per puzzle in <out-dir>/summary.txt: its id, the\n// shape-color table it learned, and the first program the reading said. Runs\n// nothing on the GPU; arc_said is what runs the programs. Run it from the repo\n// root, through scripts\\probe.bat:\n//   arc_probe <out-dir> [id...]\n// scripts\\readouts.py reads what it writes.\n#include "16_solver_beam.hpp"\n#include "14_puzzle_loader.hpp"\n#include <algorithm>\n#include <cstdio>\n#include <filesystem>\n#include <string>\n#include <vector>\nusing namespace arc;\nusing namespace std;\n\nint main(int argc, char** argv) {\n    if (argc < 2) { fprintf(stderr, "arc_probe <out-dir> [id...]\\n"); return 2; }\n    const vector<string> only(argv + 2, argv + argc);\n    task_set set = load_task_set(DS_TRAIN);\n    const filesystem::path out = argv[1];\n    filesystem::create_directories(out);\n    filesystem::current_path(out);\n    FILE* summary = fopen("summary.txt", "w");\n    if (!summary) { fprintf(stderr, "cannot write %s\\n", (out / "summary.txt").string().c_str()); return 1; }\n    int idx = 0;\n    for (const auto& entry : set.tasks) {\n        const int at = idx++;\n        if (!only.empty() && ranges::find(only, entry.id) == only.end()) continue;\n        puzzle task;\n        if (!load_task_from_content(entry.content, entry.id, task, set.solutions_for(entry.id))) continue;\n        infer_into_puzzle(task, { at, (int)set.tasks.size(), DS_TRAIN }, true);\n        const auto& rule = task.inf.shape_color_map;\n        fprintf(summary, "%s\\tshape %d/%d", entry.id.c_str(), (int)rule.claimed_by, rule.count);\n        for (int i = 0; i < rule.count; ++i) fprintf(summary, " %u>%d", rule.keys[i], rule.colors[i]);\n        fprintf(summary, "\\t%s\\n", task.suggested_programs.empty() ? "-" : task.suggested_programs.front().c_str());\n    }\n    fclose(summary);\n    return 0;\n}\n'}
for name, content in files.items():
    target = base / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
(base / 'kaggle').mkdir(exist_ok=True)
(base / 'kaggle/input').symlink_to('/kaggle/input', target_is_directory=True)
print(subprocess.check_output(['g++', '--version'], text=True))
subprocess.run([sys.executable, str(base/'scripts/cloud_setup.py')], cwd=base, check=True)
ids = ['00576224', '44d8ac46', '140c817e']
expected = ['00576224\tshape 0/0\tInput SelectAll Tile[0] Withhold CurrentObjects SelectAll InvertSelection SelectRow[1] FlipHorizontal SelectAll Union', '140c817e\tshape 0/0\tInput SelectAll Split[0] Withhold CurrentObjects SelectAll Trim Withhold CurrentObjects SelectAll BidiColorMap[9] Union', '44d8ac46\tshape 2/10 2933135872>0 4151398049>2 2701600439>0 4254437720>2 1728917041>2 2941330494>2 795915171>0 1659129079>0 2808857400>0 1630663534>2\tInput SelectAll Split[0] Withhold CurrentObjects SelectAll Trim Withhold CurrentObjects SelectAll ShapeToInteriorColorMap Union']
subprocess.run([str(base/'out/cloud/arc_probe'), 'out/probe/cloud'] + ids, cwd=base, check=True)
actual = (base/'out/probe/cloud/summary.txt').read_text().splitlines()
assert sorted(actual) == sorted(expected), (actual, expected)
print('CLOUD CPU CHECK PASSED: 3 inference summaries match the Windows build')
